# RNN vs GRU vs LSTM: a controlled comparison

Two tasks, three cells, one question: **at the same parameter budget and the same training budget, which recurrent cell is better, and can the ordering be explained by optimisation mechanics rather than noise?**

| task | metric | data |
|---|---|---|
| sentiment classification | accuracy, NLL | SST-2 (GLUE) |
| character-level text generation | bits per character (BPC) | Tiny Shakespeare |

Every cell is sized to the same parameter target. Every cell sees the same data, optimiser, schedule, batch size and number of optimizer steps. Each configuration is repeated over seeds. The comparison is reported with confidence intervals and paired tests, not with a single run.

This notebook runs the `development` tier on a GPU. The CPU tiers are `tiny` and `smoke`; they are what the test suite runs.

## What is matched, exactly

For one recurrent layer with input size $I$ and hidden size $H$, the cell parameters are

$$
P_{\text{cell}} = m\,(IH + H^2 + 2H), \qquad
m = \begin{cases}
1 & \text{RNN} \
3 & \text{GRU} \
4 & \text{LSTM}
\end{cases}
$$

because a GRU unit owns three gate matrices and an LSTM unit four. Adding the embedding $VE$ and the task head, the experiment solves for the integer $H$ closest to a shared target $P^\star$:

$$
H^\star = \arg\min_{H}\;\bigl|P(H) - P^\star\bigr|.
$$

The hidden sizes therefore differ (the LSTM needs fewer units for the same parameter count), which is the point: comparing equal *hidden width* would compare unequal capacity.

Reported statistics:

* mean $\pm$ standard deviation across seeds;
* 95 % paired bootstrap interval over test examples (classification) or validation windows (generation);
* exact McNemar test on paired per-example correctness for classification, and a paired bootstrap p-value for generation;
* Holm correction across the three pairwise comparisons.

A CI that excludes zero and a corrected $p < 0.05$ supports a difference on this test set. It is not a universal ordering.

## Why gradient reach is plotted

For a vanilla RNN, the Jacobian of the final hidden state with respect to an earlier one is a product of state-to-state matrices:

$$
\frac{\partial h_T}{\partial h_t} =
\prod_{k=t+1}^{T} \frac{\partial h_k}{\partial h_{k-1}},
$$

so the gradient norm decays roughly like a power of the largest singular value. LSTM has a more direct path through the cell state, $c_T = \prod f_k \, c_t + \cdots$, which can preserve information when the forget gates stay near one. The diagnostic measures

$$
g_t = \left\| \frac{\partial \lVert h_T \rVert^2}{\partial h_t} \right\|
$$

on the trained recurrent stack and plots it against lag $T-t$ on a log scale. A more negative fitted slope means the final state sees less far back. This is a measurement that can falsify the mechanistic story; it is not itself a proof of generalisation.

In [ ]:
import os, sys, subprocess
os.chdir("/kaggle/working")
if not os.path.isdir("deep-learning-laboratory"):
    !git clone https://github.com/Tqhuyen/deep-learning-laboratory.git
os.chdir("deep-learning-laboratory")
!git pull --ff-only
!python -m pip install -e . --quiet
!python data/prepare_sst2.py

In [ ]:
import torch
print("torch", torch.__version__)
print("device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("memory GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else "-")

## 1. Sentiment classification (SST-2)

The script runs 3 cells × 3 seeds at matched parameters. It prints validation loss and accuracy at every evaluation interval, then writes the report, raw per-example predictions, checkpoints and four figures.

In [ ]:
!python experiments/sequence_classification/sentiment/run.py --tier development --device cuda --output-directory /kaggle/working/runs

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

base = Path("/kaggle/working/runs/sequence_classification/sentiment/development")
display(Markdown((base / "report.md").read_text(encoding="utf-8")))
for name in ["validation_loss.png", "test_accuracy.png", "gradient_decay.png", "accuracy_by_length.png"]:
    display(Image(filename=str(base / name)))

## 2. Character-level text generation (Tiny Shakespeare)

Same grid, same matching logic, same optimiser budget in optimizer steps. The metric is bits per character:

$$
\mathrm{BPC} = \frac{\mathrm{CE}}{\ln 2}.
$$

Samples use the same prompt, temperature, top-k and sampling seed for every cell, so the comparison is not confounded by different random draws.

In [ ]:
!python experiments/language_modeling/recurrent_generation/run.py --tier development --device cuda --output-directory /kaggle/working/runs

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

base = Path("/kaggle/working/runs/language_modeling/recurrent_generation/development")
display(Markdown((base / "report.md").read_text(encoding="utf-8")))
for name in ["validation_bpc.png", "final_bpc.png", "gradient_decay.png"]:
    display(Image(filename=str(base / name)))
display(Markdown((base / "samples.txt").read_text(encoding="utf-8")))

## 3. Combined summary

The next cell prints the two reports' metric tables, parameter counts and fitted gradient-decay slopes. Those are the numbers to quote; the plots are the visual form of the same data.

In [ ]:
import json
from pathlib import Path

roots = {
    "sentiment": Path("/kaggle/working/runs/sequence_classification/sentiment/development"),
    "generation": Path("/kaggle/working/runs/language_modeling/recurrent_generation/development"),
}
for task, root in roots.items():
    summary = json.loads((root / "metrics.json").read_text())["summary"]
    print(f"\n== {task} ==")
    print("parameters:", summary["parameters"])
    print("metrics:", summary["metrics"])
    print("gradient slopes:", summary["gradient_slopes"])
    print("pairwise:", summary["pairwise"])
    print((root / "report.md").read_text())

## How to read the result

1. **Check the match.** The parameter counts in each report should be within a fraction of a percent of the shared target. If they are not, the comparison is not the one you think it is.
2. **Check the uncertainty.** A difference inside the bootstrap interval is not evidence; a large seed spread is also a result.
3. **Check the mechanism.** Look at the gradient-decay plot and the fitted slopes. If the cell with the longest reach is also the one with the best BPC, the explanation is consistent; if not, the mechanism story is incomplete.
4. **Check the task.** Sentiment sentences are short; generation contexts are 128 characters. A long-range advantage may not appear at these lengths. That is a boundary condition of the claim, not a failure.
5. **Do not over-claim.** These experiments establish an ordering on these distributions and budgets. They do not prove RNN, GRU or LSTM is universally better.

## References

* Hochreiter & Schmidhuber, *Long Short-Term Memory* (1997).
* Cho et al., *Learning Phrase Representations using RNN Encoder–Decoder* (2014).
* Pascanu, Mikolov, Bengio, *On the difficulty of training recurrent neural networks* (2013).
* Vaswani et al., *Attention Is All You Need* (2017).
* GLUE / SST-2 benchmark.